## Multi-Agent Orchestration with Claude Managed Agents

### Installing Utilities and Libraries

In [ ]:
%pip install anthropic==0.120.2 python-dotenv==1.2.2

### Setting up the Environment

In [ ]:
import os
from dotenv import load_dotenv

load_dotenv()

claude_model_name = os.getenv("CLAUDE_MODEL_NAME")
claude_api_key = os.getenv("CLAUDE_API_KEY")

### Creating the Anthropic Client

In [ ]:
import anthropic

client = anthropic.Anthropic(api_key = claude_api_key)

### Create the Researcher Agent

In [ ]:
researcher_agent = client.beta.agents.create(
    name="Researcher-Agent",
    model=claude_model_name,
    system="""You are a knowledgeable researcher. Your task is to gather information and provide insights on a given topic.
              You should use reliable sources and present the information in a clear and concise manner.""",
    tools=[
        {"type": "agent_toolset_20260401"},
    ],
)

print(f"Agent ID: {researcher_agent.id}, version: {researcher_agent.version}")

### Create the Writer Agent

In [ ]:
writer_agent = client.beta.agents.create(
    name="Writer-Agent",
    model=claude_model_name,
    system="""You are a creative writer. Your task is to write an essay on a given topic.
              You should focus on clarity, coherence, and engaging storytelling""",
)

print(f"Agent ID: {writer_agent.id}, version: {writer_agent.version}")

### Create the Coordinator Agent

In [ ]:
coordinator_agent = client.beta.agents.create(
    name="Editorial-Head",
    model=claude_model_name,
    system="""You are the Editorial Head Agent which whill coordinate and delegate work to the 
              researcher agent for researching topics and the writer agent for synthesizing the topics researched
              in a refined output which could be an article, essay etc. """,
    tools=[
        {"type": "agent_toolset_20260401"},
    ],
    multiagent={
        "type": "coordinator",
        "agents": [
            {"type": "agent", "id": researcher_agent.id},
            {"type": "agent", "id": writer_agent.id}
        ]
    }
)

print(f"Agent ID: {coordinator_agent.id}, version: {coordinator_agent.version}")

### Create the Environment

In [ ]:
environment = client.beta.environments.create(
    name="quickstart-env",
    config={
        "type": "cloud",
        "networking": {"type": "unrestricted"},
    },
)

print(f"Environment ID: {environment.id}")

### Create the Session

In [ ]:
session = client.beta.sessions.create(
    agent=coordinator_agent.id,
    environment_id=environment.id,
)

### Execute the Agent

In [ ]:
with client.beta.sessions.events.stream(session.id) as stream:
            # Send the user message after the stream opens
            client.beta.sessions.events.send(
                session.id,
                events=[
                    {
                        "type": "user.message",
                        "content": [
                            {
                                "type": "text",
                                "text": """write an essay about the impact of AI on society
                                           Don't perform a lot of research, use only one instance of researcher agent
                                           and rather make it quick pls. Keep the essay really short under 250 words pls""",
                            },
                        ],
                    },
                ],
            )

            # Process streaming events
            for event in stream:
                match event.type:
                    case "agent.message":
                        for block in event.content:
                            print(block.text, end="")
                    case "agent.tool_use":
                        print(f"\n[Using tool: {event.name}]")
                    case "session.status_idle":
                        print("\n\nAgent finished.")
                        break